# Mixture of Experts (MoE) Implementation - Solutions

This notebook contains reference solutions for the MoE interview questions.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Question 1: Basic MoE Implementation

In [ ]:
class MLP(nn.Module):
    """Single expert network."""
    
    def __init__(self, hidden_size, ffn_hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, ffn_hidden_size)
        self.fc2 = nn.Linear(ffn_hidden_size, hidden_size)
        self.activation = nn.GELU()
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x


class MoE(nn.Module):
    """Mixture of Experts layer."""
    
    def __init__(self, num_experts, hidden_size, ffn_hidden_size, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # Router network
        self.router = nn.Linear(hidden_size, num_experts)
        
        # Expert networks
        self.experts = nn.ModuleList([
            MLP(hidden_size, ffn_hidden_size) for _ in range(num_experts)
        ])
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        batch_size, seq_len, hidden_size = x.shape
        
        # Step 1: Get routing scores
        router_logits = self.router(x)  # (B, S, num_experts)
        
        # Step 2: Apply softmax to get routing probabilities
        router_probs = F.softmax(router_logits, dim=-1)  # (B, S, num_experts)
        
        # Step 3: Select top-k experts and their weights
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # top_k_probs: (B, S, top_k)
        # top_k_indices: (B, S, top_k)
        
        # Normalize the top-k probabilities
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        # Step 4-5: Process through experts and combine
        # Flatten batch and sequence dimensions for easier processing
        x_flat = x.view(-1, hidden_size)  # (B*S, H)
        output = torch.zeros_like(x_flat)  # (B*S, H)
        
        # Flatten the top_k tensors
        top_k_probs_flat = top_k_probs.view(-1, self.top_k)  # (B*S, top_k)
        top_k_indices_flat = top_k_indices.view(-1, self.top_k)  # (B*S, top_k)
        
        # For each expert, find tokens routed to it and process them
        for expert_idx in range(self.num_experts):
            # Find which tokens route to this expert
            expert_mask = (top_k_indices_flat == expert_idx)  # (B*S, top_k)
            token_indices, k_indices = expert_mask.nonzero(as_tuple=True)
            
            if len(token_indices) == 0:
                continue
            
            # Get the tokens that route to this expert
            expert_input = x_flat[token_indices]  # (num_tokens, H)
            
            # Process through expert
            expert_output = self.experts[expert_idx](expert_input)  # (num_tokens, H)
            
            # Get the routing weights for these tokens
            weights = top_k_probs_flat[token_indices, k_indices].unsqueeze(-1)  # (num_tokens, 1)
            
            # Add weighted expert output to the final output
            output[token_indices] += weights * expert_output
        
        # Reshape back to original dimensions
        output = output.view(batch_size, seq_len, hidden_size)
        
        return output

### Question 1a: Test the Implementation

In [ ]:
# Test basic functionality
x = torch.randn(2, 3, 5)  # (batch=2, seq_len=3, hidden=5)
moe = MoE(num_experts=4, hidden_size=5, ffn_hidden_size=20, top_k=2)

print("Input shape:", x.shape)
output = moe(x)
print("Output shape:", output.shape)
assert output.shape == x.shape, "Output shape should match input shape"

# Check for NaN/Inf
assert not torch.isnan(output).any(), "Output contains NaN"
assert not torch.isinf(output).any(), "Output contains Inf"

# Test gradient flow
loss = output.sum()
loss.backward()

# Check that router has gradients
assert moe.router.weight.grad is not None, "Router should have gradients"
print("Router gradient norm:", moe.router.weight.grad.norm().item())

# Check that experts have gradients
for i, expert in enumerate(moe.experts):
    assert expert.fc1.weight.grad is not None, f"Expert {i} should have gradients"

print("\nAll tests passed!")

## Question 2: Load Balancing Strategies

### Answer:

**Strategy 1: Auxiliary Load Balancing Loss**
- Add a differentiable loss term that penalizes uneven expert usage
- Measure: fraction of tokens assigned to each expert (hard assignment) vs. mean routing probability (soft assignment)
- Tradeoff: Small loss coefficient may not balance well; large coefficient may hurt model quality

**Strategy 2: Capacity Factor with Token Dropping**
- Limit the maximum number of tokens each expert can process (capacity = tokens_per_batch / num_experts × capacity_factor)
- Drop tokens that exceed expert capacity
- Tradeoff: Guarantees balance but may lose information from dropped tokens

**Strategy 3: Expert Choice Routing**
- Instead of tokens choosing experts, have experts choose their top-k tokens
- Each expert processes exactly the same number of tokens
- Tradeoff: Perfect balance but changes routing dynamics; may be less intuitive

**Detection:**
- Monitor the distribution of token assignments across experts
- Track metrics like coefficient of variation or entropy of expert usage
- Log expert utilization during training

## Question 2a: Implement Auxiliary Load Balancing Loss

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_size, ffn_hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, ffn_hidden_size)
        self.fc2 = nn.Linear(ffn_hidden_size, hidden_size)
        self.activation = nn.GELU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x


class MoE(nn.Module):
    def __init__(self, num_experts, hidden_size, ffn_hidden_size, top_k=2, aux_loss_coef=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.aux_loss_coef = aux_loss_coef
        
        self.router = nn.Linear(hidden_size, num_experts)
        self.experts = nn.ModuleList([
            MLP(hidden_size, ffn_hidden_size) for _ in range(num_experts)
        ])
    
    def forward(self, x):
        # x: (batch_size, seq_len, hidden_size)
        batch_size, seq_len, hidden_size = x.shape
        
        # Get routing scores and probabilities
        router_logits = self.router(x)  # (B, S, num_experts)
        router_probs = F.softmax(router_logits, dim=-1)  # (B, S, num_experts)
        
        # Select top-k experts
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        # Process through experts (same as before)
        x_flat = x.view(-1, hidden_size)
        output = torch.zeros_like(x_flat)
        
        top_k_probs_flat = top_k_probs.view(-1, self.top_k)
        top_k_indices_flat = top_k_indices.view(-1, self.top_k)
        
        for expert_idx in range(self.num_experts):
            expert_mask = (top_k_indices_flat == expert_idx)
            token_indices, k_indices = expert_mask.nonzero(as_tuple=True)
            
            if len(token_indices) == 0:
                continue
            
            expert_input = x_flat[token_indices]
            expert_output = self.experts[expert_idx](expert_input)
            weights = top_k_probs_flat[token_indices, k_indices].unsqueeze(-1)
            output[token_indices] += weights * expert_output
        
        output = output.view(batch_size, seq_len, hidden_size)
        
        # Calculate auxiliary load balancing loss
        if self.training:
            # f_i: fraction of tokens assigned to expert i (hard assignment)
            # Count how many tokens are routed to each expert
            expert_counts = torch.zeros(self.num_experts, device=x.device)
            for expert_idx in range(self.num_experts):
                expert_mask = (top_k_indices_flat == expert_idx)
                expert_counts[expert_idx] = expert_mask.sum().float()
            
            # Normalize to get fraction
            total_tokens = batch_size * seq_len * self.top_k
            f = expert_counts / total_tokens  # (num_experts,)
            
            # P_i: mean routing probability for expert i (soft assignment)
            # Average the routing probabilities across all tokens
            P = router_probs.mean(dim=[0, 1])  # (num_experts,)
            
            # Auxiliary loss: α * num_experts * sum(f_i * P_i)
            aux_loss = self.aux_loss_coef * self.num_experts * (f * P).sum()
            
            return output, aux_loss
        
        return output

In [ ]:
# Test code
x = torch.randn(2, 3, 5)
moe = MoE(num_experts=4, hidden_size=5, ffn_hidden_size=20, top_k=2, aux_loss_coef=0.01)
moe.train()

output, aux_loss = moe(x)
print(f"Output shape: {output.shape}")
print(f"Auxiliary loss: {aux_loss.item():.4f}")

# Verify aux_loss is differentiable
total_loss = output.sum() + aux_loss
total_loss.backward()
print(f"Router gradient norm: {moe.router.weight.grad.norm().item():.4f}")

# Test eval mode (should return only output)
moe.eval()
output_eval = moe(x)
assert isinstance(output_eval, torch.Tensor), "Eval mode should return only output"
print("\n✓ Auxiliary loss implementation correct!")

## Question 3: Expert Parallelism with Data Parallelism

### Part A: Communication Operations Solution

In [ ]:
class MoE(nn.Module):
    def __init__(self, num_experts, hidden_size, ffn_hidden_size, world_size=2, rank=0):
        super().__init__()
        self.num_experts = num_experts
        self.world_size = world_size
        self.rank = rank
        
        # Router is shared across all GPUs (replicated)
        self.router = nn.Linear(hidden_size, num_experts)
        
        # Each GPU stores a subset of experts
        # For example, with 4 experts and 2 GPUs:
        # GPU 0: experts [0, 1]
        # GPU 1: experts [2, 3]
        experts_per_rank = num_experts // world_size
        start_expert = rank * experts_per_rank
        self.local_experts_ids = list(range(start_expert, start_expert + experts_per_rank))
        
        self.local_experts = nn.ModuleList([
            MLP(hidden_size, ffn_hidden_size) 
            for _ in self.local_experts_ids
        ])
    
    def forward(self, x):
        # x: (local_batch_size, seq_len, hidden_size) - only local samples
        # local_batch_size = global_batch_size // world_size
        local_batch_size, seq_len, hidden_size = x.shape
        global_batch_size = local_batch_size * self.world_size
        
        # Get routing decisions using local data
        scores = self.router(x)  # (local_B, S, num_experts)
        probs = F.softmax(scores, dim=-1)
        routing_weights, expert_ids = probs.max(dim=-1)  # (local_B, S)
        
        # Step 1: Gather expert assignments from all GPUs
        # Operation: all_gather (gather tensors from all ranks)
        # Each GPU has expert_ids of shape (local_B, S)
        # We need to gather from all GPUs to get shape (global_B, S)
        global_expert_ids = torch.empty(
            (global_batch_size, seq_len),
            dtype=expert_ids.dtype,
            device=expert_ids.device
        )
        torch.distributed.all_gather_into_tensor(
            global_expert_ids,  # output: (global_B, S)
            expert_ids          # input from this GPU: (local_B, S)
        )
        
        # Step 2: Gather input tensors from all GPUs
        # Operation: all_gather
        # Each GPU has x of shape (local_B, S, H)
        # We need to gather to get shape (global_B, S, H)
        global_x = torch.empty(
            (global_batch_size, seq_len, hidden_size),
            dtype=x.dtype,
            device=x.device
        )
        torch.distributed.all_gather_into_tensor(
            global_x,  # output: (global_B, S, H)
            x          # input from this GPU: (local_B, S, H)
        )
        
        # Step 3: Each GPU processes tokens assigned to its local experts
        output_total = torch.zeros_like(global_x)
        
        for i, expert in enumerate(self.local_experts):
            local_expert_id = self.local_experts_ids[i]
            
            # Find tokens routed to this expert
            mask = (global_expert_ids == local_expert_id)
            expert_input = global_x[mask]
            
            # Process with expert
            expert_output = expert(expert_input)
            
            # Store in output
            output_total[mask] = expert_output
        
        # Step 4: Reduce outputs across all GPUs then split back to local
        # First, we need to sum outputs from all GPUs (all_reduce)
        # Then split to get local portion (this is implicit in all_reduce)
        # Actually, for MoE we want to gather all outputs then scatter back
        # So we use all_reduce to sum contributions, then divide by world_size
        
        # Alternative approach: Use reduce_scatter
        # reduce_scatter: reduce (sum) and then scatter the result
        # But PyTorch all_reduce followed by indexing is simpler
        
        # Operation: all_reduce (sum outputs across GPUs)
        torch.distributed.all_reduce(output_total, op=torch.distributed.ReduceOp.SUM)
        
        # Now extract only the local portion
        # GPU 0 gets samples [0:local_batch_size]
        # GPU 1 gets samples [local_batch_size:2*local_batch_size]
        start_idx = self.rank * local_batch_size
        end_idx = start_idx + local_batch_size
        output_local = output_total[start_idx:end_idx]  # (local_B, S, H)
        
        # Apply routing weights
        output_local = output_local * routing_weights.unsqueeze(-1)
        
        return output_local

### Detailed Explanation of Communication Operations:

**Step 1 & 2: `all_gather`**
- **Purpose**: Collect expert assignments and input tensors from all GPUs
- **Input**: Each GPU has its local portion (local_batch_size samples)
- **Output**: All GPUs have the complete data (global_batch_size samples)
- **Why**: Tokens on GPU 0 might need experts on GPU 1, so all GPUs need all data

**Step 3: Local Computation**
- Each GPU processes only tokens assigned to its local experts
- No communication needed here

**Step 4: `all_reduce` (sum operation)**
- **Purpose**: Combine expert outputs from all GPUs
- **Input**: Each GPU has processed some tokens (output_total with many zeros)
- **Output**: Sum of all outputs, so each position has the output from whichever GPU processed it
- **Why**: Each token was processed on exactly one GPU (the one with its assigned expert), we need to combine results
- **Then**: Extract local portion by indexing

**Alternative using `reduce_scatter`:**
Could use `reduce_scatter` which combines `all_reduce` + scatter in one operation, but the above is clearer.

### Part B: Load Balancing and GPU Efficiency

**Answer:**

Load balancing is critical for GPU utilization in expert-parallel setups:

**Problem with Imbalance:**
- If one expert gets 90% of tokens:
  - The GPU hosting that expert becomes a bottleneck
  - Other GPUs sit idle waiting for the overloaded GPU
  - Memory usage is imbalanced (one GPU may OOM while others are underutilized)
  - Communication costs increase (more data transferred to one GPU)

**Impact on Training:**
- **Compute**: Training speed is limited by the slowest GPU (stragglers)
- **Memory**: Imbalanced memory usage can cause OOM errors even with sufficient total memory
- **Communication**: All-to-all communication becomes bottlenecked by the overloaded GPU
- **Throughput**: Overall throughput drops to the throughput of the busiest GPU

**Benefits of Load Balancing:**
- All GPUs process similar amounts of work → better parallelism
- More predictable memory usage → can use larger batch sizes
- Reduced communication time → less waiting
- Near-linear scaling with number of GPUs

**Practical Impact:**
- With perfect balance: 8 GPUs → ~8x speedup
- With 90% imbalance: 8 GPUs → ~1.1x speedup (most GPUs idle)
- Load balancing loss (even with small coefficient like 0.01) typically improves both model quality and training efficiency

## Summary

Key concepts covered:

1. **Basic MoE**: Router network + top-k routing + weighted expert combination
2. **Load Balancing**: Auxiliary loss to encourage even expert usage
3. **Distributed Training**: 
   - Data parallelism splits batches across GPUs
   - Expert parallelism splits experts across GPUs
   - Requires all_gather (gather data) and all_reduce (combine outputs)
   - Load balancing prevents GPU stragglers

These are fundamental building blocks for large-scale MoE models like Switch Transformer, GLaM, and others.